In [11]:
# Verify "U.S. Avocado Market Trends" Memo
# Cross-checks every figure in `US_Avocado_Market_Trends_Memo.docx` against
# the two raw weekly source files (`2025.xls`, `2026PartYear.xls`).

# Design: one pass builds a single tidy DataFrame restricted to the
# comparable 32-week window in both years, then every memo figure is
# derived from vectorized `groupby` aggregations on that one frame —
# no per-figure re-filtering of the raw files.

import pandas as pd

PATH_2025 = "2025.csv"           # full year 2025 (52 weeks)
PATH_2026 = "2026PartYear.csv"   # year-to-date 2026

TOL_PP = 0.15   # tolerance, in percentage points, before flagging a mismatch

In [12]:
# 1. Load & build one comparable dataset

def load(path: str, year: int) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["Current Year Week Ending"] = pd.to_datetime(df["Current Year Week Ending"])
    df["Year"] = year
    return df

raw25 = load(PATH_2025, 2025)
raw26 = load(PATH_2026, 2026)

n_weeks = raw26["Period"].max()          # 32, per the 2026 file
last_week = raw26["Current Year Week Ending"].max().date()

comp25 = raw25[raw25["Period"] <= n_weeks]   # first N weeks of 2025 only
data = pd.concat([comp25, raw26], ignore_index=True)

# volume-weighted ASP needs sum(ASP*volume) before it can be averaged
data["asp_x_vol"] = data["ASP Current Year"] * data["Total Bulk and Bags"]

print(f"Comparable window: first {n_weeks} weeks of each year "
      f"(2026 YTD through {last_week}); "
      f"{len(comp25)} 2025 rows + {len(raw26)} 2026 rows loaded.")

Comparable window: first 32 weeks of each year (2026 YTD through 2026-08-09); 4096 2025 rows + 4096 2026 rows loaded.


In [13]:
# 2. One grouped table drives every downstream figure

agg = (
    data.groupby(["Year", "Type", "Geography"], observed=True)
    .agg(
        volume=("Total Bulk and Bags", "sum"),
        bagged=("TotalBagged", "sum"),
        asp_x_vol=("asp_x_vol", "sum"),
    )
    .reset_index()
)
agg["vw_asp"] = agg["asp_x_vol"] / agg["volume"]          # volume-weighted ASP
agg["bagged_pct"] = agg["bagged"] / agg["volume"] * 100    # % bagged

def pivot_pct_change(geo: str, ptype: str, col: str):
    """Return (2025 value, 2026 value, % change) for one Geography/Type/metric."""
    sub = agg[(agg["Geography"] == geo) & (agg["Type"] == ptype)].set_index("Year")[col]
    v25, v26 = sub.loc[2025], sub.loc[2026]
    return v25, v26, (v26 - v25) / v25 * 100

def check(label, actual_pct, memo_pct):
    ok = abs(actual_pct - memo_pct) <= TOL_PP
    flag = "OK" if ok else "MISMATCH"
    print(f"{flag:9s} {label:45s} memo={memo_pct:+6.1f}%  actual={actual_pct:+6.2f}%")
    return ok

In [14]:
# 3. Executive Summary / National Market Trends figures

results = []

v25, v26, pct = pivot_pct_change("Total U.S.", "Conventional", "volume")
print(f"Conventional volume: {v25:,.0f} (2025) -> {v26:,.0f} (2026), "
      f"+{v26 - v25:,.0f} units")
results.append(check("Conventional volume % change", pct, 14.5))

a25, a26, pct = pivot_pct_change("Total U.S.", "Conventional", "vw_asp")
print(f"Conventional ASP: ${a25:.2f} (2025) -> ${a26:.2f} (2026)")
results.append(check("Conventional ASP % change", pct, -19.4))

v25, v26, pct = pivot_pct_change("Total U.S.", "Organic", "volume")
print(f"Organic volume: {v25:,.0f} (2025) -> {v26:,.0f} (2026)")
results.append(check("Organic volume % change", pct, -0.7))

a25, a26, pct = pivot_pct_change("Total U.S.", "Organic", "vw_asp")
print(f"Organic ASP: ${a25:.2f} (2025) -> ${a26:.2f} (2026)")
results.append(check("Organic ASP % change", pct, -10.3))

Conventional volume: 1,764,839,344 (2025) -> 2,020,684,608 (2026), +255,845,264 units
OK        Conventional volume % change                  memo= +14.5%  actual=+14.50%
Conventional ASP: $1.22 (2025) -> $0.98 (2026)
OK        Conventional ASP % change                     memo= -19.4%  actual=-19.44%
Organic volume: 105,859,521 (2025) -> 105,115,569 (2026)
OK        Organic volume % change                       memo=  -0.7%  actual= -0.70%
Organic ASP: $1.74 (2025) -> $1.56 (2026)
OK        Organic ASP % change                          memo= -10.3%  actual=-10.28%


In [15]:
# 4. Product Mix figures (bagged share of volume)

b25, b26, _ = pivot_pct_change("Total U.S.", "Conventional", "bagged_pct")
print(f"Conventional bagged share: {b25:.1f}% (2025) -> {b26:.1f}% (2026), "
      f"+{b26 - b25:.1f} pp")
results.append(check("Conventional bagged share, 2025", b25, 39.5))
results.append(check("Conventional bagged share, 2026", b26, 41.3))

org_bagged_2026 = agg.query("Geography=='Total U.S.' and Type=='Organic' and Year==2026")["bagged_pct"].iloc[0]
results.append(check("Organic bagged share, 2026", org_bagged_2026, 81.3))

Conventional bagged share: 39.5% (2025) -> 41.3% (2026), +1.8 pp
OK        Conventional bagged share, 2025               memo= +39.5%  actual=+39.47%
OK        Conventional bagged share, 2026               memo= +41.3%  actual=+41.29%
OK        Organic bagged share, 2026                    memo= +81.3%  actual=+81.30%


In [16]:
# 5. Regional / Metro conventional-volume growth

memo_regions = {
    "Raleigh/Greensboro": 23.9,
    "Miami/FT. Lauderdale": 23.7,   # memo spells it "Miami/Fort Lauderdale"
    "New Orleans/Mobile": 23.3,
    "Orlando": 21.9,
    "Baltimore/Washington": 20.5,
    "Southeast": 20.7,
    "Seattle": 1.9,
    "Portland": 3.1,
}

for geo, memo_pct in memo_regions.items():
    _, _, pct = pivot_pct_change(geo, "Conventional", "volume")
    results.append(check(f"{geo} conventional volume % change", pct, memo_pct))

OK        Raleigh/Greensboro conventional volume % change memo= +23.9%  actual=+23.87%
OK        Miami/FT. Lauderdale conventional volume % change memo= +23.7%  actual=+23.72%
OK        New Orleans/Mobile conventional volume % change memo= +23.3%  actual=+23.28%
OK        Orlando conventional volume % change          memo= +21.9%  actual=+21.91%
OK        Baltimore/Washington conventional volume % change memo= +20.5%  actual=+20.46%
OK        Southeast conventional volume % change        memo= +20.7%  actual=+20.73%
OK        Seattle conventional volume % change          memo=  +1.9%  actual= +1.85%
OK        Portland conventional volume % change         memo=  +3.1%  actual= +3.08%


In [17]:
# 6. Summary

n_ok = sum(results)
print(f"\n{n_ok}/{len(results)} memo figures confirmed within ±{TOL_PP} pp of the source data.")
if n_ok < len(results):
    print("Review the MISMATCH lines above.")


15/15 memo figures confirmed within ±0.15 pp of the source data.


In [18]:
# Create a timestamp and author
from datetime import datetime
time_now=datetime.now().strftime('%m_%d_%Y_%I%M%p')
print("Current Time:{}".format(time_now))
print("Joshua Lira")

Current Time:09_24_2026_0834PM
Joshua Lira
